# MANOS A LA OBRA

A continuación, explicamos y desarrollamos paso a paso el ejercicio “Ruta a la electrificación de la movilidad”, donde aprenderemos técnicas para el tratamiento, visualización y análisis avanzado de datos abiertos mediante Python. Antes de empezar, recomendamos leer el apartado [Readme](https://github.com/Admindatosgobes/Laboratorio-de-Datos/tree/main/Data%20Science/Ruta%20a%20la%20electrificaci%C3%B3n%20de%20la%20Movilidad).

Comenzaremos realizando una preparación sencilla de nuestro entorno de trabajo, posteriormente realizaremos una etapa de volcado y preparación de los datos, para finalmente proceder a su análisis utilizando técnicas de visualización y analítica avanzada.

## 🎛1. Configuración Inicial

Con el objetivo de establecer las bases para desarrollar nuestro trabajo, realizaremos en primer lugar dos tareas principales:
* Importamos todas las librerías de Python que utilizaremos a lo largo de este Notebook.

In [ ]:
# Librerías             ---------------------------------------------
# Utilidades
import requests
import os
import csv
import PyPDF2
import zipfile
import datetime
from dateutil.relativedelta import relativedelta
import warnings
warnings.filterwarnings('ignore')
# Manipulación de datos
import numpy as np
import pandas as pd # Still used for compatibility in some models if needed
import polars as pl
import codecs
# Visualización de datos
import seaborn as sns
import matplotlib.pyplot as plt
# Modelos estadísticos
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf


In [1]:
import os
import numpy as np
import pandas as pd # Still used for compatibility in some models if needed
import polars as pl
import statsmodels.api as sm
import pmdarima as pm

In [2]:
os.chdir("..")
os.getcwd()

'/Users/juanjose/Library/CloudStorage/GoogleDrive-jj.rincon@student.ie.edu/My Drive/Iberdrola Datathon'

In [3]:
# Variables principales ---------------------------------------------
dir_data_standardized = 'data/standardized/'
file_registrations = os.path.join(dir_data_standardized, 'vehicle_registrations.parquet')

print(f"Cargando datos desde: {file_registrations}")


Cargando datos desde: data/standardized/vehicle_registrations.parquet


## ⬇ 2. Preparación de Datos

In [ ]:
# Importamos utilidades y sincronizamos datos
import sys
import os

# Aseguramos que el directorio raíz está en el path
if os.getcwd().endswith('notebooks'):
    os.chdir('..')

if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from scripts.sync_cloud import sync_standardized_data

# Synchronize data from cloud (ensure silver layer is present)
sync_standardized_data()

print("\n✅ Datos sincronizados. Utilizando capa de estandarización (Silver Layer).")

Como podemos observar, la carpeta *Data/zip*, antes vacía, se encuentra ahora repleta de ficheros .zip, uno por cada mes consultado.


## 📊3. Análisis de Datos

Procedemos en este apartado del ejercicio al análisis de los datos descargados. Para ello trataremos de contestar a preguntas mediante visualizaciones de datos y análisis estadístico.

Tras revisar las variables disponibles, decidimos utilizar las siguientes para nuestro análisis:
* FEC_MATRICULA: Fecha en la que se produjo la matriculación.
* MARCA_ITV: Fabricante del vehículo.
* MODELO_ITV: Modelo del vehículo.
* COD_TIPO: Código del tipo de vehículo (i.e. camión, turismo, apisonadora...).
* COD_PROPULSION_ITV: Código del tipo de propulsión (i.e. Gasolina, Diésel, Eléctrico...).
* CATEGORÍA_VEHÍCULO_ELÉCTRICO: Categoría de vehículo eléctrico (i.e. Eléctrico Enchufable, Eléctrico Híbrido, Eléctrico de Batería...).
* CLAVE TRAMITE: Código del trámite (i.e. Matriculación, Transferencia, Baja Temporal...).

Inicialmente, deseamos disponer de los datos en un Dataframe para poder trabajar con ellos. Para ello, cargamos los datos de matriculaciones desde los ficheros *parquet* anteriormente generados utilizando la librería *pandas*.

In [4]:
from datetime import date

df = pl.read_parquet(file_registrations).filter(
   ( pl.col('date') >= date(2015, 1, 1) ) & (pl.col('date') < date(2026, 1, 1)) 
)
df

date,brand,propulsion
date,str,str
2015-01-02,"""VOLKSWAGEN""","""Diesel"""
2015-01-02,"""VOLKSWAGEN""","""Diesel"""
2015-01-02,"""LEXUS""","""Gasolina"""
2015-01-02,"""BMW""","""Diesel"""
2015-01-02,"""AUDI""","""Diesel"""
…,…,…
2025-12-30,"""HYUNDAI""","""Gasolina"""
2025-12-31,"""VOLKSWAGEN""","""Diesel"""
2025-11-10,"""AUDI""","""Diesel"""


In [ ]:
non_ev_registrations = df.filter(pl.col('propulsion') != 'Eléctrico').group_by(
    pl.col("date").dt.truncate("1mo")
).agg(
    pl.col('brand').count().alias('registrations')    
).sort("date")

ev_registrations = df.filter(pl.col('propulsion') == 'Eléctrico').group_by(
    pl.col("date").dt.truncate("1mo")
).agg(
    pl.col('brand').count().alias('registrations')
).sort("date")


date,registrations
date,u32
2015-01-01,71240
2015-02-01,90257
2015-03-01,115720
2015-04-01,86170
2015-05-01,97666
…,…
2025-08-01,64418
2025-09-01,86796
2025-10-01,102718


In [17]:
ev_registrations = ev_registrations.to_pandas().set_index('date')
ev_registrations

,registrations
date,
2015-01-01,55
2015-02-01,51
2015-03-01,162
2015-04-01,111
2015-05-01,118
...,...
2025-08-01,7809
2025-09-01,11016
2025-10-01,10097


In [24]:
# Selección automática del modelo con Auto-ARIMA
auto_EV_model = pm.auto_arima(np.log(ev_registrations['registrations']),
                      test='adf',        # ADF Test
                      m=12,              # Estacionalidad de 12 meses
                      seasonal=True,     # Modelo SARIMA
                      trace=False,
                      error_action='ignore',
                      suppress_warnings=True,
                      stepwise=True)


# Ajuste del modelo SARIMAX con los parámetros óptimos de auto_arima
best_order_EV = auto_EV_model.order
best_seasonal_order_EV = auto_EV_model.seasonal_order

print(f"Ajustando SARIMAX con order={best_order_EV} y seasonal_order={best_seasonal_order_EV}")

EV_model = sm.tsa.statespace.SARIMAX(
    np.log(ev_registrations['registrations']), 
    order=best_order_EV, 
    seasonal_order=best_seasonal_order_EV
)
EV_results = EV_model.fit(disp=False)

forecast_steps = 24
EV_model_forecast = EV_results.get_forecast(steps=forecast_steps)

EV_model_forecast = (EV_model_forecast.summary_frame(alpha=0.5))
EV_fc_mean = round(np.exp(EV_model_forecast['mean']))

# Crear DataFrame para el pronóstico
last_date = ev_registrations.index.max()
forecast_dates = [last_date + relativedelta(months=i) for i in range(1, forecast_steps + 1)]

df_ev_forecast = pd.DataFrame({
    'registrations': EV_fc_mean
}, index=forecast_dates)


ev_registrations = pd.concat([ev_registrations, df_ev_forecast]).rename('ev_registrations')

ev_registrations

Ajustando SARIMAX con order=(1, 0, 2) y seasonal_order=(1, 0, 1, 12)


/Users/juanjose/Library/CloudStorage/GoogleDrive-jj.rincon@student.ie.edu/My Drive/Iberdrola Datathon/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/Users/juanjose/Library/CloudStorage/GoogleDrive-jj.rincon@student.ie.edu/My Drive/Iberdrola Datathon/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/Users/juanjose/Library/CloudStorage/GoogleDrive-jj.rincon@student.ie.edu/My Drive/Iberdrola Datathon/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


TypeError: 'str' object is not callable

In [ ]:
# Selección automática del modelo con Auto-ARIMA
auto_EV_model = pm.auto_arima(np.log(ev_registrations['registrations']),
                      test='adf',        # ADF Test
                      m=12,              # Estacionalidad de 12 meses
                      seasonal=True,     # Modelo SARIMA
                      trace=False,
                      error_action='ignore',
                      suppress_warnings=True,
                      stepwise=True)


# Ajuste del modelo SARIMAX con los parámetros óptimos de auto_arima
best_order_EV = auto_EV_model.order
best_seasonal_order_EV = auto_EV_model.seasonal_order

print(f"Ajustando SARIMAX con order={best_order_EV} y seasonal_order={best_seasonal_order_EV}")

EV_model = sm.tsa.statespace.SARIMAX(
    np.log(ev_registrations['registrations']), 
    order=best_order_EV, 
    seasonal_order=best_seasonal_order_EV
)
EV_results = EV_model.fit(disp=False)

forecast_steps = 24
EV_model_forecast = EV_results.get_forecast(steps=forecast_steps)

EV_model_forecast = (EV_model_forecast.summary_frame(alpha=0.5))
EV_fc_mean = round(np.exp(EV_model_forecast['mean']))

# Crear DataFrame para el pronóstico
last_date = ev_registrations.index.max()
forecast_dates = [last_date + relativedelta(months=i) for i in range(1, forecast_steps + 1)]

df_ev_forecast = pd.DataFrame({
    'registrations': EV_fc_mean
}, index=forecast_dates)


ev_registrations = pd.concat([ev_registrations, df_ev_forecast]).rename('ev_registrations')

ev_registrations

,registrations
2015-01-01,55.0
2015-02-01,51.0
2015-03-01,162.0
2015-04-01,111.0
2015-05-01,118.0
...,...
2027-08-01,11609.0
2027-09-01,16695.0
2027-10-01,16779.0
2027-11-01,17671.0


ALTERNATIVA

Adicionalmente, trabajamos sobre nuestro conjunto de datos para:
* Disponer de dos columnas adicionales con el mes y el año de la matriculación.
* Eliminar del conjunto de datos todas aquellas matriculaciones que no corresponden a vehículos de tipo turismo o no son matriculaciones en firme.
* Sustituimos los códigos de propulsión por sus descripciones.

En los siguientes apartados, utilizaremos herramientas de visualización y análisis de datos para poder responder a cuestiones sobre el conjunto de datos seleccionado para este ejercicio.

### 1) ¿Qué marcas de vehículos lideraron el mercado en 2025?

Para responder a esta pregunta, excluimos del análisis todas aquellas matriculaciones que corresponden a matriculaciones no realizadas en el año 2025. Posteriormente, realizamos un conteo de vehículos por marca.

In [ ]:
df_1 = (
    df.filter(pl.col('ANO') == 2025)
    .group_by('MARCA_ITV')
    .agg(pl.len().alias('MATRICULACIONES'))
    .sort('MATRICULACIONES', descending=True)
)
df_1.head(5).to_pandas()


In [ ]:
sns.set(rc={'figure.figsize':(20,8)})
sns.barplot(x='MARCA_ITV', y='MATRICULACIONES', data=df_1)
plt.xticks(rotation=90);

Dado que disponemos de muchas marcas, vamos a realizar esta misma visualización sólo para las Top 10 marcas que más vehículos vendieron este año, para poder visualizar mejor los resultados.

In [ ]:
sns.barplot(x='MARCA_ITV', y='MATRICULACIONES', data=df_1.head(10))
plt.title('TOP 10 MARCAS 2025')
plt.xticks(rotation=90);

Observamos cómo Toyota y Volkswagen son las marcas líderes con un volumen de matriculaciones de turismos superior a 100.000 unidades en 2025. Vemos cómo en el Top 10 predominan marcas asiáticas y europeas, no existiendo ninguna china ni americana.

### 2) ¿Qué modelos de vehículos fueron los más vendidos en el 2025?

Para responder a esta pregunta, excluimos del análisis todas aquellas matriculaciones que corresponden a matriculaciones no realizadas en el año 2023. Posteriormente, realizamos un conteo de vehículos por marca y modelo, pero también distinguiremos entre el tipo de propulsión del vehículo, de tal forma que veamos si entre los más matriculados aparece algún vehículo eléctrico.

In [ ]:
# Análisis de matriculaciones por marca, modelo y propulsión (Polars)
df_2 = (
    df.filter(pl.col('ANO') == 2025)
    .group_by(['MARCA_ITV', 'MODELO_ITV', 'COD_PROPULSION_ITV'])
    .agg(pl.len().alias('MATRICULACIONES'))
    .sort('MATRICULACIONES', descending=True)
)

df_2.head(10).to_pandas()

In [ ]:
sns.barplot(x='MODELO_ITV', y='MATRICULACIONES', data=df_2.head(10), hue='COD_PROPULSION_ITV')
plt.title('TOP 10 VEHÍCULOS 2025')
plt.xticks(rotation=90);

Tras realizar esta visualización, aparecen algunos aspectos interesantes:
* Los únicos vehículos de fabricación europea que aparecen en el Top 10 son el Arona y el Ibiza de la marca española SEAT. El resto son asiáticos.
* Nueve de los diez vehículos están propulsados por Gasolina.
* El único vehículo del Top 10 con un tipo de propulsión diferente es el DACIA Sandero GLP (Gas Licuado de Petróleo).

### 3) ¿Qué cuota de mercado absorbieron los vehículos eléctricos en el 2025?

Para responder a esta pregunta, centraremos nuestro análisis en las tres categorías principales de propulsión de vehículos: Gasolina, Diésel y Eléctrico. Clasificaremos el resto de tecnologías en la catergoría "Otros" y analizaremos a través de un gráfico de tarta el peso actual de las matriculaciones de VEs en el último año.

In [ ]:
# Análisis de propulsión agrupando categorías minoritarias en 'Otros' (Polars)
main_propulsions = ['Gasolina', 'Diesel', 'Eléctrico']

df_3 = (
    df.filter(pl.col('ANO') == 2025)
        .with_columns(
            COD_PROPULSION_ITV=pl.when(pl.col('COD_PROPULSION_ITV').is_in(main_propulsions))
            .then(pl.col('COD_PROPULSION_ITV')).otherwise(pl.lit('Otros'))
        )
        .group_by('COD_PROPULSION_ITV')
        .agg(pl.len().alias('MATRICULACIONES'))
        .sort('MATRICULACIONES', descending=True)
        )

df_3.head(5).to_pandas()

In [ ]:
colores = ['#4F6272', '#B7C3F3', '#DD7596', '#8EB897'] # Colores para el gráfico
plt.pie(df_3['MATRICULACIONES'], labels=df_3['COD_PROPULSION_ITV'], autopct='%1.1f%%', colors=colores) # Gráfico de tipo tarta
plt.title('VEHÍCULOS POR TIPO DE PROPULSIÓN 2025')
plt.axis('equal');

Vemos cómo la inmensa mayoría del mercado (>70%) la absorbieron vehículos de gasolina, siendo los diésel la segunda opción, y como los vehículos eléctricos alcanzaron el 8.6%.

### 4) ¿Qué modelos de vehículos eléctricos fueron los más vendidos en el 2025?

Para responder a esta pregunta, agrupamos las matriculaciones por marca y modelo de vehículo, filtrando todas aquellas matriculaciones que no sean del 2025 ni de vehículos eléctricos.

In [ ]:
# Análisis de modelos eléctricos líderes en 2025 (Polars)
df_4 = (
    df.filter((pl.col('ANO') == 2025) & (pl.col('COD_PROPULSION_ITV') == 'Eléctrico'))
    .group_by(['MARCA_ITV', 'MODELO_ITV'])
    .agg(pl.len().alias('MATRICULACIONES'))
    .sort('MATRICULACIONES', descending=True)
)

df_4.head(10).to_pandas()

In [ ]:
sns.barplot(x='MODELO_ITV', y='MATRICULACIONES', data=df_4.head(10), hue='MARCA_ITV')
plt.title('TOP 10 VEHÍCULOS ELÉCTRICOS 2025')
plt.xticks(rotation=90);

Podemos observar en la gráfica anterior cómo TESLA es la empresa cuyos modelos lideran el ranking, siendo el Model Y y el Model 3 los más matriculados con una amplia diferencia frente al tercer vehículo eléctrico: el MG4.

### 5) ¿Cómo han evolucionado las matriculaciones de vehículos a lo largo del tiempo?

Comenzamos ahora a analizar la evolución de las matriculaciones de vehículos en el tiempo. Este tipo de análisis se considera un análisis de **series temporales**, pues disponemos de conjunto de observaciones recogidas o registradas en intervalos de tiempo consecutivos o regulares. La característica distintiva de una serie temporal es que el orden de estas observaciones es importante, lo que significa que hay una dependencia clara del tiempo en los datos.

Comenzaremos analizando la evolución global de matriculaciones para posteriormente analizar por tipo de propulsión.

In [ ]:
# Evolución mensual total de matriculaciones (Polars)
df_5_1 = (
    df.group_by(['ANO', 'MES'])
    .agg(pl.len().alias('MATRICULACIONES'))
    .sort(['ANO', 'MES'])
)

df_5_1.head().to_pandas()

In [ ]:
# Representamos en un diagrama de barras
df_5_1.to_pandas()['MATRICULACIONES'].plot(kind='bar')
plt.title('MATRICULACIONES DE VEHÍCULOS');

Podemos observar varios aspectos interesantes en este gráfico:
* Apreciamos un **comportamiento estacional** anual, es decir, observamos patrones o variaciones que se repiten a intervalos regulares de tiempo. Vemos cómo recurrentemente en junio/julio aparecen altos niveles de matriculación mientras que en agosto/septiembre decrecen drásticamente. Esto es muy relevante, pues el análisis de series temporales con factor estacional tiene ciertas particularidades.
* Es muy notable también la enorme caída de matriculaciones producida durante los primeros meses del COVID.
* Por último, vemos como los niveles de matriculación post-covid son inferiores a los previos.

Procederemos ahora a desarrollar el mismo análisis contemplado el tipo de propulsión.

In [ ]:
# Agrupación por tipo de propulsión y tiempo (Polars + Pivot)
main_propulsions = ['Gasolina', 'Diesel', 'Eléctrico']

df_5_2_pl = (
    df.with_columns(
        COD_PROPULSION_ITV=pl.when(pl.col('COD_PROPULSION_ITV').is_in(main_propulsions))
        .then(pl.col('COD_PROPULSION_ITV'))
        .otherwise(pl.lit('Otros'))
    )
    .group_by(['ANO', 'MES', 'COD_PROPULSION_ITV'])
    .agg(pl.len().alias('MATRICULACIONES'))
)

# Pivotamos y convertimos a Pandas para representación visual
df_5_2 = (
    df_5_2_pl.pivot(on='COD_PROPULSION_ITV', index=['ANO', 'MES'], values='MATRICULACIONES')
    .fill_null(0)
    .sort(['ANO', 'MES'])
    .to_pandas()
    .set_index(['ANO', 'MES'])
)

df_5_2.head()

In [ ]:
# Representamos el número de vehículos matriculados por tipo de propulsion a lo largo del tiempo
df_5_2[['Gasolina','Diesel','Eléctrico', 'Otros']].plot(kind='bar',stacked=True)
plt.title('MATRICULACIONES DE VEHÍCULOS POR TIPO PROPULSIÓN');

Podemos observar rápidamente como entre los años 2015 y 2025 la matriculación de vehículos eléctricos va creciendo paulatinamente. Para observarlo de forma más clara, procederemos a realizar una visualización de los ratios de matriculación por tipo de propulsión.

In [ ]:
# Creamos una copia del dataframe anteriore
df_5_3 = df_5_2.copy()

# Calculamos el número total de matriculaciones por mes
df_5_3['TOTAL'] = df_5_3.iloc[:, 0:4].sum(axis=1)

#Calculamos el porcentaje de matriculaciones por tipo de propulsión cada mes
for columna in df_5_3.columns[0:4]:
    df_5_3[columna + '_%'] = (df_5_3[columna] / df_5_3['TOTAL']) * 100
df_5_3.head()

In [ ]:
# Representamos en un diagrama de barras
df_5_3[['Gasolina_%','Diesel_%','Eléctrico_%','Otros_%']].plot(kind='bar', stacked=True);
plt.title('MATRICULACIONES DE VEHÍCULOS POR TIPO PROPULSIÓN');

Vemos ahora de forma mucho más sencilla el efecto antes descrito. Es también notable en este gráfico el detrimento de vehículos diésel en favor de vehículos de tipo gasolina.

### 6) ¿Observamos algún tipo de tendencia respecto a la matriculación de vehículos eléctricos?

Para responder esta pregunta, analizaremos por separado la evolución de vehículos eléctricos y no eléctricos utilizando mapas de calor como herramienta visual.

In [ ]:
# Preparación de datos para mapas de calor (Polars + Pivot)

def get_heatmap_data(df_base):
    return (
        df_base.group_by(['ANO', 'MES'])
        .agg(pl.len().alias('MATRICULACIONES'))
        .pivot(on='ANO', index='MES', values='MATRICULACIONES')
        .fill_null(0)
        .sort('MES')
        .to_pandas()
        .set_index('MES')
        .astype(float)
    )

df_6_1 = get_heatmap_data(df.filter(pl.col('COD_PROPULSION_ITV') != 'Eléctrico'))
df_6_2 = get_heatmap_data(df.filter(pl.col('COD_PROPULSION_ITV') == 'Eléctrico'))

df_6_1.head(12)

In [ ]:
# Creamos una figura con dos espacios para gráficos
fig, axs = plt.subplots(1, 2)

# Representamos ambos mapas de calor
sns.heatmap(df_6_1, ax=axs[0])
sns.heatmap(df_6_2, ax=axs[1])
axs[0].set_title('MATRICULACIONES DE VEHÍCULOS CONVENCIONALES')
axs[1].set_title('MATRICULACIONES DE VEHÍCULOS ELÉCTRICOS');

Podemos observar comportamientos muy diferenciados entre ambos gráficos. Observamos cómo el vehículo eléctrico presenta una tendencia de incremento de matriculaciones año a año y, a pesar de suponer el COVID un parón en la matriculación de vehículos, los años posteriores han mantenido la tendencia creciente.



# 7) ¿Cómo esperamos que evolucionen las matriculaciones de Vehículos en los próximos 2 años?

## 7.1) ¿Cómo esperamos que evolucionen las matriculaciones de Vehículos Electricos en los próximos 2 años?

Las respuestas anteriores nos han permitido familiarizarnos con el conjunto de datos y entender mejor los comportamientos de las matriculaciones de vehículos, entendiendo aspectos relevantes como el componente estacional y la tendencia alzista del VE. Esta pregunta nos requerirá un análisis más avanzado: debemos realizar una previsión futura.

En primer lugar, visualicemos la serie temporal de matriculaciones de VEs.

In [ ]:
# Filtrado para vehículos eléctricos y agregación mensual con Polars
pf_EV = (
    df.filter(pl.col('COD_PROPULSION_ITV') == 'Eléctrico')
    .group_by(['ANO', 'MES'])
    .agg(pl.len().alias('MATRICULACIONES'))
    .with_columns(
        FECHA = pl.date(pl.col('ANO'), pl.col('MES'), 1)
    )
    .sort('FECHA')
)

# Convertimos a Pandas para compatibilidad con statsmodels
df_EV = pf_EV.to_pandas().set_index('FECHA')
df_EV = df_EV[['MATRICULACIONES']]

print(f"Serie temporal preparada. Rango: {df_EV.index.min()} a {df_EV.index.max()}")
df_EV.head()

In [ ]:
# Representamos los datos ob
fig,ax = plt.subplots(figsize=(12,8),dpi=100)
df_EV['MATRICULACIONES'].plot(title="Matriculaciones de Vehículos Eléctricos")
ax.set_ylabel('MATRICULACIONES');

Varios son los componentes que integran una serie temporal:
* Tendencia: Es el componente que muestra el movimiento a largo plazo de los datos, indicando un aumento o disminución general a lo largo del tiempo.
* Estacionalidad: Es el componente que muestra variaciones regulares y predecibles que ocurren en un período específico, como diario, semanal, mensual o trimestral.
* Irregularidad: También conocido como componente aleatorio o ruido, es el componente que no sigue ningún patrón regular y puede ser resultado de factores aleatorios o inesperados. Este componente es impredecible y varía de manera aleatoria a lo largo del tiempo.

En primer lugar, trataremos de detectar algún tipo de tendencia de forma visual. Para ello, calculamos la media móvil anual de las matriculaciones de vehículos eléctricos. Ideal para suavizar la estacionalidad anual, permitiendo observar la tendencia subyacente en los datos eliminando las fluctuaciones estacionales.

In [ ]:
# Calculamos la media móvil
media_movil = df_EV.rolling(
    window=12,       # Ventana de 12 meses (anual)
    center=True,
    min_periods=6,
).mean()

# Mostramos su representación para analizar su comportamiento
fig,ax = plt.subplots(figsize=(12,8),dpi=100)
df_EV.plot(ax=ax, style=".", color="0.5", legend=False)
media_movil.plot( ax=ax, linewidth=3, title="Media Móvil Anual", legend=False)
ax.set_ylabel('MATRICULACIONES');

Como podemos observar, al suavizar la representación gracias a la media móvil hay una tendencia creciente clara en matriculaciones de vehículo eléctrico.

Además del análisis visual, podemos utilizar tests estadísticos para determinar la estacionariedad de la serie temporal estudiada de forma cuantitativa.

Uno de los tests más utilizados para este fin es el [**Augmented Dickey-Fuller (ADF)**](https://es.wikipedia.org/wiki/Prueba_de_Dickey-Fuller_aumentada), test de tipo raíz unitaria. Un test de raíz unitaria es una prueba estadística que determina si una serie temporal es estacionaria alrededor de una tendencia o es diferencialmente estacionaria, es decir, si contiene una raíz unitaria. Hay varias pruebas de raíz unitaria y la de ADF es una de las más utilizadas. El planteamiento de este test es el siguiente:
* Hipótesis Nula (H0): Si no se rechaza, sugiere que la serie temporal tiene una raíz unitaria, significando que es no estacionaria, y por tanto tiene alguna estructura dependiente del tiempo.
* Hipótesis Alternativa (H1): La hipótesis nula es rechazada sugiriendo que la serie temporal no tiene una raíz unitaria y por tanto es estacionaria. No tiene estructura dependiente del tiempo.

Para interpretar este resultado, utilizaremos el valor p (*p-value*) de la prueba. Un valor p por debajo de un umbral (5%) sugiere que rechazamos la hipótesis nula y por tanto la serie es estacionaria. Por el contrario, un valor p por encima del umbral sugiere que no rechazamos la hipótesis nula y por tanto la serie no es estacionaria.

In [ ]:
# Augmented Dicky Fuller Test (ADF)
def adf_test(serie_temporal):
  resultado = adfuller(serie_temporal)
  adf_estadistico = resultado[0]
  valor_p = resultado[1]
  print('ADF Estadístico: %f' %adf_estadistico)
  print('p-value: %f' %valor_p)
  return adf_estadistico, valor_p


In [ ]:
adf_statistic, p_value = adf_test(df_EV['MATRICULACIONES'])

Nuestro test ADF muestra un p-value elevado, muy superior al valor umbral (0,05), indicando por tanto que la serie no es estacionaria, exhibe algún tipo de tendencia como ya observamos anteriormente de forma visual.

En último lugar, procedemos a realizar la predicción de matriculaciones de vehículo eléctrico para el próximo año. Para ello, utilizaremos el modelo ARIMA estacional o **SARIMA**. El modelo SARIMA, que significa "Seasonal AutoRegressive Integrated Moving Average", es una extensión del modelo ARIMA diseñada para capturar tanto las tendencias y patrones no estacionales como las fluctuaciones estacionales en series temporales. Combina los conceptos de modelos autorregresivos (AR), integración (I) para hacer la serie estacionaria, y medias móviles (MA), junto con componentes estacionales equivalentes (SAR, SI, SMA) para modelar efectivamente datos que exhiben patrones estacionales recurrentes. Se especifica mediante los parámetros (p, d, q)(P, D, Q)m, donde p, d, q representan la parte no estacional y P, D, Q, m la parte estacional del modelo. Es ampliamente utilizado para el análisis y pronóstico de series temporales complejas que presentan tanto tendencias a largo plazo como ciclos estacionales.



Nos apoyaremos en la librería *pmdarima* para realizar de forma automatizada la configuración de este modelo adecuado a nuestro conjunto de datos.

In [ ]:
# Selección automática del modelo con Auto-ARIMA
auto_EV_model = pm.auto_arima(np.log(df_EV['MATRICULACIONES']),
                      test='adf',        # ADF Test
                      m=12,              # Estacionalidad de 12 meses
                      seasonal=True,     # Modelo SARIMA
                      trace=True,
                      error_action='ignore',
                      suppress_warnings=True,
                      stepwise=True)

print(auto_EV_model.summary())


Con los resultados anteriormente obtenidos, creamos un modelo SARIMA y realizamos la predicción para los siguientes 12 meses.

In [ ]:
# Ajuste del modelo SARIMAX con los parámetros óptimos de auto_arima
best_order_EV = auto_EV_model.order
best_seasonal_order_EV = auto_EV_model.seasonal_order

print(f"Ajustando SARIMAX con order={best_order_EV} y seasonal_order={best_seasonal_order_EV}")

EV_model = sm.tsa.statespace.SARIMAX(
    np.log(df_EV['MATRICULACIONES']), 
    order=best_order_EV, 
    seasonal_order=best_seasonal_order_EV
)
EV_results = EV_model.fit(disp=False)

print(EV_results.summary())


In [ ]:
# Generación de pronóstico a 24 meses y exportación consolidada
forecast_steps = 24
EV_model_forecast = EV_results.get_forecast(steps=forecast_steps)

EV_model_forecast = (EV_model_forecast.summary_frame(alpha=0.5))

EV_fc_mean = round(np.exp(EV_model_forecast['mean']))
EV_fc_lb = round(np.exp(EV_model_forecast['mean_ci_lower']))
EV_fc_ub = round(np.exp(EV_model_forecast['mean_ci_upper']))


# Representamos predicción
fig,ax = plt.subplots(figsize=(12,8),dpi=100)
plt.plot(df_EV.index,df_EV['MATRICULACIONES'])
fc_months = [datetime.datetime(2025,11,1) + relativedelta(months=x) for x in range(24)]
plt.plot(fc_months, EV_fc_mean, label = 'forecast_media', linewidth=1.5)
plt.legend(loc='upper left')
plt.fill_between(fc_months, EV_fc_lb, EV_fc_ub,color='b',alpha=0.1, label = '95% Confianza')
ax.set_title('EV Registrations Forecast 2027')
ax.set_ylabel('Registrations')


In [ ]:
# Crear DataFrame para el pronóstico
last_date = df_EV.index.max()
forecast_dates = [last_date + relativedelta(months=i) for i in range(1, forecast_steps + 1)]

df_ev_forecast = pd.DataFrame({
    'ev_registrations': EV_fc_mean
}, index=forecast_dates)

ev_registrations = df_EV.rename(columns={'MATRICULACIONES':'ev_registrations'})

ev_registrations = pd.concat([ev_registrations, df_ev_forecast])

ev_registrations

# Non Electric Vehicles Registration Prediction

In [ ]:
pf_non_ev = (
    df.filter(pl.col('COD_PROPULSION_ITV') != 'Eléctrico')
    .group_by(['ANO', 'MES'])
    .agg(pl.len().alias('MATRICULACIONES'))
    .with_columns(
        FECHA = pl.date(pl.col('ANO'), pl.col('MES'), 1)
    )
    .sort('FECHA')
)

# Convertimos a Pandas para compatibilidad con statsmodels
df_non_ev = pf_non_ev.to_pandas().set_index('FECHA')
df_non_ev = df_non_ev[['MATRICULACIONES']]

print(f"Serie temporal preparada. Rango: {df_non_ev.index.min()} a {df_non_ev.index.max()}")

# Representamos los datos ob
fig,ax = plt.subplots(figsize=(12,8),dpi=100)
df_non_ev['MATRICULACIONES'].plot(title="Non Electric Vehicles Registrations")
ax.set_ylabel('Registrations');

In [ ]:
adf_statistic, p_value = adf_test(df_non_ev['MATRICULACIONES'])

In [ ]:
# Selección automática del modelo con Auto-ARIMA
non_ev_model = pm.auto_arima(np.log(df_non_ev['MATRICULACIONES']),
                      test='adf',        # ADF Test
                      m=12,              # Estacionalidad de 12 meses
                      seasonal=True,     # Modelo SARIMA
                      trace=True,
                      error_action='ignore',
                      suppress_warnings=True,
                      stepwise=True)

print(non_ev_model.summary())


In [ ]:
# Ajuste del modelo SARIMAX con los parámetros óptimos de auto_arima
best_order = non_ev_model.order
best_seasonal_order = non_ev_model.seasonal_order

print(f"SARIMAX with order={best_order}, seasonal_order={best_seasonal_order} & intercept={non_ev_model.with_intercept} ")

non_ev_model = sm.tsa.statespace.SARIMAX(
    np.log(df_non_ev['MATRICULACIONES']), 
    trend='c' if non_ev_model.with_intercept==True else 'n',
    order=best_order, 
    seasonal_order=best_seasonal_order
)
non_ev_results = non_ev_model.fit(disp=False)

print(non_ev_results.summary())


In [ ]:
# Generación de pronóstico a 24 meses y exportación consolidada
non_ev_model_forecast = non_ev_results.get_forecast(steps=forecast_steps)

non_ev_model_forecast = (non_ev_model_forecast.summary_frame(alpha=0.5))

non_ev_fc_mean = round(np.exp(non_ev_model_forecast['mean']))
non_ev_fc_lb = round(np.exp(non_ev_model_forecast['mean_ci_lower']))
non_ev_fc_ub = round(np.exp(non_ev_model_forecast['mean_ci_upper']))

# Representamos predicción
fig,ax = plt.subplots(figsize=(12,8),dpi=100)
plt.plot(df_non_ev.index,df_non_ev['MATRICULACIONES'])
plt.plot(fc_months, non_ev_fc_mean, label = 'forecast_media', linewidth=1.5)
plt.legend(loc='upper left')
plt.fill_between(fc_months, non_ev_fc_lb, non_ev_fc_ub,color='b',alpha=0.1, label = '95% Confianza')
ax.set_title('Non EV Registrations Forecast 2027')
ax.set_ylabel('Registrations')


In [ ]:
# Crear DataFrame para el pronóstico
last_date = df_non_ev.index.max()

df_non_ev_forecast = pd.DataFrame({
    'non_ev_registrations': non_ev_fc_mean
}, index=forecast_dates)

non_ev_registrations = df_non_ev.rename(columns={'MATRICULACIONES':'non_ev_registrations'})

non_ev_registrations = pd.concat([non_ev_registrations, df_non_ev_forecast])

non_ev_registrations

In [ ]:
vehicle_registrations = ev_registrations.merge(non_ev_registrations,how='inner',left_index=True, right_index=True)
vehicle_registrations

In [ ]:
vehicle_registrations['total_registrations'] = vehicle_registrations['ev_registrations'] + vehicle_registrations['non_ev_registrations']
vehicle_registrations['ev_share'] = vehicle_registrations['ev_registrations'] / vehicle_registrations['total_registrations']
vehicle_registrations

In [ ]:
total_ev = vehicle_registrations['ev_registrations'].sum()
total_non_ev = vehicle_registrations['non_ev_registrations'].sum()
total_all = total_ev + total_non_ev
total_ev/total_all

2027 vehicles

In [ ]:
v_2027 = vehicle_registrations.loc[vehicle_registrations.index>='2027-01-01']
v_2027

In [ ]:
total_ev = v_2027['ev_registrations'].sum()
total_non_ev = v_2027['non_ev_registrations'].sum()
total_all = total_ev + total_non_ev
print(f'total electric vehicles in 2027: {total_ev}, market share {round((total_ev/total_all)*100,2)}%')

In [ ]:
v_2025 = vehicle_registrations.loc[(vehicle_registrations.index>='2025-01-01')&(vehicle_registrations.index<'2026-01-01')]
total_ev = v_2025['ev_registrations'].sum()
total_non_ev = v_2025['non_ev_registrations'].sum()
total_all = total_ev + total_non_ev
print(f'total electric vehicles in 2025: {total_ev}, market share {round((total_ev/total_all)*100,2)}%')

In [ ]:
vehicle_registrations.to_parquet('data/processed/ev_registrations_forecast.parquet')